In [ ]:
import numpy as np
import pandas as pd
import copy
import matplotlib.pyplot as plt
import seaborn as sns
import shap
from scipy import stats, interpolate
from sklearn.model_selection import train_test_split, GridSearchCV, KFold,  cross_val_score, StratifiedKFold, cross_validate
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_curve, roc_auc_score, RocCurveDisplay, auc, balanced_accuracy_score,  precision_recall_curve, confusion_matrix, ConfusionMatrixDisplay, average_precision_score
from sklearn.calibration import calibration_curve, CalibrationDisplay
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import mutual_info_classif, SelectPercentile
from xgboost import XGBClassifier, XGBRFClassifier
from lightgbm import LGBMClassifier
from sklearn.preprocessing import StandardScaler, LabelBinarizer, MinMaxScaler
from scipy.stats import levene, ttest_ind, ttest_rel, chi2, chi2_contingency,chisquare
import pingouin as pg
from imblearn.over_sampling import SMOTE, SMOTEN,KMeansSMOTE#, SMOTETomek
import pickle
from bayes_opt import BayesianOptimization
import xgboost as xgb
import math, random

In [ ]:
plt.rc('font', size= 12)          # controls default text sizes
plt.rc('axes', titlesize= 15)     # fontsize of the axes title
plt.rc('axes', labelsize= 15)    # fontsize of the x and y labels
plt.rc('xtick', labelsize=15)    # fontsize of the tick labels
plt.rc('ytick', labelsize=15)    # fontsize of the tick labels
plt.rc('legend', fontsize=12)    # legend fontsize
plt.rc('figure', titlesize=15)  # fontsize of the figure title

In [ ]:
import imblearn, sklearn, xgboost, lightgbm, bayes_opt, scipy
print(sklearn.__version__)
print(bayes_opt.__version__)
print(shap.__version__)
print(xgboost.__version__)
print(lightgbm.__version__)
print(imblearn.__version__)
print(scipy.__version__)

# Parameter tuning

## XGBoost

In [ ]:
def XGB_cv(max_depth,learning_rate, n_estimators, gamma
           ,min_child_weight, max_delta_step, subsample
           ,colsample_bytree, nthread=-1):
           
    model = XGBClassifier(max_depth=int(max_depth),
                              learning_rate=learning_rate,
                              n_estimators=int(n_estimators),
                              nthread=nthread,
                              gamma=gamma,
                              min_child_weight=min_child_weight,
                              max_delta_step=max_delta_step,
                              subsample=subsample,
                              colsample_bytree=colsample_bytree,
                              verbosity = 2)

    cross_score = cross_val_score(model, train_x, train_y, cv=SK_cv).mean()    
    return cross_score

In [ ]:
def tuning_xgb() :
    pbounds = {'max_depth': (5, 10),
            'learning_rate': (0.01, 0.3),
            'n_estimators': (50, 1000),
            'gamma': (0.01, 1.),
            'min_child_weight': (2, 10),
            'max_delta_step': (0, 0.1),
            'subsample': (0.7, 0.8),
            'colsample_bytree' :(0.5, 0.99)
            }

    xgboostBO = BayesianOptimization(f = XGB_cv,pbounds = pbounds, verbose = 2, random_state = 42)

    xgboostBO.maximize(init_points= 2, n_iter = n_iters)

    xgb_best_params = xgboostBO.max 
    with open(f'renewal 1/{feature_name})_xgb_parameters.pickle','wb') as fw:
        pickle.dump(xgb_best_params, fw)

    fit_xgb = XGBClassifier(max_depth= int( xgboostBO.max['params']['max_depth'] ),
                             learning_rate=xgboostBO.max['params']['learning_rate'],
                             n_estimators=int(xgboostBO.max['params']['n_estimators']),
                             gamma= xgboostBO.max['params']['gamma'],
                             min_child_weight=xgboostBO.max['params']['min_child_weight'],
                             max_delta_step=xgboostBO.max['params']['max_delta_step'],
                             subsample=xgboostBO.max['params']['subsample'],
                             colsample_bytree=xgboostBO.max['params']['colsample_bytree'])
    return fit_xgb

## LGBM

In [ ]:
def LGBM_cv(max_depth, num_leaves, min_child_samples, min_child_weight, subsample, colsample_bytree, max_bin, reg_lambda, reg_alpha) :
    params = {
        "n_estimators" : 500, "learning_rate": 0.02,
        'max_depth': int(round(max_depth)),
        'num_leaves': int(round(num_leaves)),
        'min_child_samples': int(round(min_child_samples)),
        'min_child_weight': int(round(min_child_weight)),
        'subsample': max(min(subsample, 1), 0),
        'colsample_bytree': max(min(colsample_bytree, 1), 0),
        'max_bin':  max(int(round(max_bin)),10),
        'reg_lambda': max(reg_lambda,0),
        'reg_alpha': max(reg_alpha, 0)
    }
    model = LGBMClassifier(**params, verbose = 0)
    cross_score = cross_val_score(model, train_x, train_y, cv=SK_cv).mean()  
    return cross_score

In [ ]:
def tuning_lgbm() :
    pbounds = {
        'max_depth':(1, 16),
        'num_leaves':(24,64),
        'min_child_samples': (10, 200),
        'min_child_weight': (1, 50), 
        'subsample': (0.5, 1), #
        'colsample_bytree': (0.5, 1),
        'max_bin': (10, 500),
        'reg_lambda': (0.001, 10), 
        'reg_alpha': (0.01, 50)
            }

    lightgbmBO = BayesianOptimization(f = LGBM_cv,pbounds = pbounds, verbose = 2, random_state = 42)
    lightgbmBO.maximize(init_points=2, n_iter = n_iters)
    lgbm_best_params = lightgbmBO.max
    fit_lgbm = LGBMClassifier(
      n_estimators = 500, learning_rate = 0.02,
      max_depth = int(round(lgbm_best_params["params"]["max_depth"])),
      num_leaves = int(round(lgbm_best_params["params"]["num_leaves"])),
      min_child_samples = int(round(lgbm_best_params["params"]["min_child_samples"])),
      min_child_weight = int(round(lgbm_best_params["params"]["min_child_weight"])),
      subsample = max(min(lgbm_best_params["params"]["subsample"], 1), 0), # 0 과 1 사이
      colsample_bytree = max(min(lgbm_best_params["params"]["colsample_bytree"], 1), 0),
      max_bin = max(int(round(lgbm_best_params["params"]["max_bin"])),10),
      reg_lambda = max(lgbm_best_params["params"]["reg_lambda"],0),
      reg_alpha = max(lgbm_best_params["params"]["reg_alpha"], 0)
                            )
    return fit_lgbm

## RF

In [ ]:
def RF_cv(max_samples, max_features, n_estimators, max_depth) :
    params = {
        'max_samples': max_samples,
        'max_features':max_features,
        'n_estimators':int(n_estimators),
        'max_depth': int(max_depth),
    }
    model = RandomForestClassifier(**params, verbose = 0) 
    cross_score = cross_val_score(model, train_x, train_y, cv=SK_cv).mean() 
    return cross_score


In [ ]:
def tuning_rf() :
    pbounds = {
            'max_samples':(0.5,1),
            'max_features':(0.5,1),
            'n_estimators':(100,200),
            'max_depth': (1,10)
            }

    rfBO = BayesianOptimization(f = RF_cv,pbounds = pbounds, verbose = 2, random_state = 42)


    rfBO.maximize(init_points=2, n_iter = n_iters)

    fit_rf = RandomForestClassifier(
                            max_samples = rfBO.max["params"]["max_samples"],
                            max_features = rfBO.max["params"]["max_features"],
                            n_estimators = int(rfBO.max["params"]["n_estimators"]),
                            max_depth = int(round(rfBO.max["params"]["max_depth"]))
                                )
    return fit_rf

## XGBRF

In [ ]:
def XGBRF_cv(num_parallel_tree, max_depth, subsample, colsample_bytree, gamma) :
    params = {
        'num_parallel_tree': int(num_parallel_tree),
        'max_depth': int(max_depth),
        'subsample' : subsample,
        "colsample_bytree" : colsample_bytree,
        "learning_rate" : 1,
        "gamma" : gamma
    }
    model = XGBRFClassifier(**params, verbosity = 0)
    cross_score = cross_val_score(model, train_x, train_y, cv=SK_cv).mean() 
    return cross_score


In [ ]:
def tuning_xgbrf() :
    pbounds = {
        'num_parallel_tree': (100,500),
        'max_depth': (4,10),
        'subsample' : (0.5,1.0),
        "colsample_bytree" : (0.5,1.0),
        "gamma" : (0,0.5)
            }

    xgbrfBO = BayesianOptimization(f = XGBRF_cv,pbounds = pbounds, verbose = 2, random_state = 42)


    xgbrfBO.maximize(init_points=2, n_iter = n_iters)

    xgbrf_best_params = xgbrfBO.max
    fit_xgbrf = XGBRFClassifier(
                            num_parallel_tree = int(xgbrfBO.max["params"]['num_parallel_tree']),
                            max_depth = int(xgbrfBO.max["params"]['max_depth']),
                            subsample = xgbrfBO.max["params"]['subsample'],
                            colsample_bytree = xgbrfBO.max["params"]["colsample_bytree"],
                            learning_rate = 1,
                            gamma = xgbrfBO.max["params"]["gamma"]
                                )
    
    return fit_xgbrf

# Plottinig function for Feature importance(shap)

In [ ]:
def FI_grouping(df) :
    grouped_df = df.copy()
    wearable_keywords = ["HR", "steps", "L5", "M10", "IV", "IS_", "RA", "dist", "exercise", "sleep", "wake", "REM"]
    smartphone_keywords = ["naptime", "stress", "alcohol", "caffeine"]
    pathiology_keywords = ["Age", "Sex", "BMI"]
    def assign_group(var):
        var_lower = var
        if any(keyword in var_lower for keyword in wearable_keywords):
            return "Wearable Device"
        elif any(keyword in var_lower for keyword in smartphone_keywords):
            return "Smartphone App"
        elif any(keyword in var_lower for keyword in pathiology_keywords) :
            return "Demographic Information"

        else:
            return "Self-Questionnaire"
    grouped_df['group'] = grouped_df['feature'].apply(assign_group)
    return grouped_df

In [ ]:
def plot_FI(x_train, y_train, x_test, model, model_name) :
    if "Random" not in model_name :
        model.fit(x_train, y_train)
        tree_explainer = shap.TreeExplainer(model)
        shap_value = tree_explainer.shap_values(x_test)
        feature_importance = pd.DataFrame({"feature" : x_test.columns,
                                "importance" : np.abs(shap_value).mean(axis=0)})
        
        feature_importance = feature_importance.sort_values(by='importance', ascending=False).reset_index(drop=True)
        grouped_feature_importance = FI_grouping(feature_importance)
        plt.figure(figsize=(10,10))
        ax = sns.barplot(data=grouped_feature_importance[:20], y='feature', x='importance', hue='group', palette=custom_palette)
        for container in ax.containers:
            ax.bar_label(container, fmt="%.2f", label_type="edge")
        ax.legend(title=None)
        plt.xlabel("SHAP value (feature importance)")
        plt.ylabel("")
        plt.tight_layout()
        plt.title(f"SHAP value for {model_name} \n using top 20 SHAP value of {columns}")

        plt.savefig(f"renewal 1/{columns}_{oversam}_{tuning}_SHAP_{model_name}.png", bbox_inches = "tight")        
    else  :

        model.fit(x_train, y_train)
        tree_explainer = shap.TreeExplainer(model)
        shap_value = tree_explainer.shap_values(x_test)
        feature_importance = pd.DataFrame({"feature" : x_test.columns,
                                "importance" : np.abs(shap_value).mean(axis=2).mean(axis=0)})

        print("X_test shape:", x_test.shape)
        print("shap_values[1] shape:", shap_value[1].shape)

        feature_importance = feature_importance.sort_values(by='importance', ascending=False).reset_index(drop=True)
        grouped_feature_importance = FI_grouping(feature_importance)
        plt.figure(figsize=(10,10))
        ax = sns.barplot(data=grouped_feature_importance[:20], x="importance", y='feature', hue='group', palette=custom_palette)
        for container in ax.containers:
            ax.bar_label(container, fmt="%.2f", label_type="edge")
        ax.legend(title=None)
        plt.xlabel("SHAP value (feature importance)")
        plt.ylabel("")
        plt.tight_layout()
        plt.title(f"SHAP value for {model_name} \n using top 20 SHAP value of {columns}")
        plt.savefig(f"renewal 1/{columns}_{oversam}_{tuning}_SHAP_{model_name}.png", bbox_inches = "tight")


In [ ]:
def model_run(x_train, y_train, x_test, y_test, model, model_names) :
    score = dict()
    n_splits = 5
    cv = StratifiedKFold(n_splits=n_splits, shuffle = True)
    smt = SMOTE(random_state=42)
    tprs = dict()
    mean_fpr = np.linspace(0, 1, 100)
    mean_recall = np.linspace(0, 1, 100)
    mean_prob = np.linspace(0, 1, 100)
    results = {n : {} for n in model_names}
    score = {n : {} for n in model_names}
    thresholds = np.linspace(0, 1, 100)
    cms = dict()
    for m, n in zip(model, model_names) :
        m.fit(x_train, y_train)
        proba = m.predict_proba(x_test)[:,1]
        pred = m.predict(x_test)
        cm = confusion_matrix(y_test, pred)
        cms[n] = cm
        prec_vals, rec_vals, _ = precision_recall_curve(y_test, proba)
        pr_auc_val = auc(rec_vals, prec_vals)
        interp_precision = np.interp(mean_recall, rec_vals[::-1], prec_vals[::-1])
        interp_precision = np.maximum.accumulate(interp_precision[::-1])[::-1]
        prob_true, prob_pred = calibration_curve(y_test, proba, n_bins=3)   
        n_total = len(y_test)
        net_benefits = []
        for t in thresholds :
            tp = np.sum((proba >= t) & (y_test == 1))
            fp = np.sum((proba >= t) & (y_test == 0))

            nb = (tp / n_total) - (fp / n_total) * (t / (1 - t)) if t < 1 else 0
            net_benefits.append(nb)
        
        fpr, tpr, _ = roc_curve(y_test, proba)
        interp_tpr = np.interp(mean_fpr, fpr, tpr)
        interp_tpr[0] = 0
        results[n] = {
                    "fpr": mean_fpr,
                    "tpr": interp_tpr,
                    "roc_auc": auc(fpr, tpr),

                    # "recall": rec_vals,
                    "recall": mean_recall,
                    # "precision": prec_vals,
                    "precision": interp_precision,
                    "pr_auc": pr_auc_val,

                    "prob_pred": prob_pred,
                    "prob_true": prob_true,

                    "net_benefit": net_benefits
                }
        score[n] = {
                    "acc" : accuracy_score(y_test,pred),
                    "prec" : precision_score(y_test,pred),
                    "recall" : recall_score(y_test, pred),
                    "f1" : f1_score(y_test, pred),
                    "auc" : auc(fpr, tpr)
                }

    for n in model_names :
        plt.figure(figsize=(10,10))
        sns.heatmap(cms[n], annot=True, fmt='d', cmap='Blues', cbar=False,
                    xticklabels=["Low Risk", "High Risk"], 
                    yticklabels=["Low Risk", "High Risk"],
                    annot_kws={"size": 15}, square=True)
        plt.ylabel('Actual', fontsize=12)
        plt.xlabel('Predicted', fontsize=12)
        plt.title(f"Confusion Matrix of {n} using {feature_name}")
        plt.tight_layout()
        plt.savefig(f"renewal 1/{feature_name}_{oversam}_{tuning}_{n}_confusion matrix.png", bbox_inches = "tight")   
    plt.figure(figsize=(10,10))
    for n in model_names :
        plt.plot(mean_fpr,
            results[n]['tpr'],
            label=f"{n} ROC (AUC = {round(score[n]['auc'], 2)})" ,
        )
    plt.plot([0,1], [0,1], "k--", label = "Chance_Level (AUC = 0.5)")

    plt.xlim([-0.05, 1.05])
    plt.ylim([-0.05, 1.05])
    plt.xlabel("False Positive Rate (FPR)")
    plt.ylabel("True Positive Rate (TPR)")
    plt.title(f"Receiver Operating Characteristics Curves of OSA Prediction Models \n using {feature_name}")
    plt.tight_layout()
    plt.legend()
    plt.savefig(f"renewal 1/{feature_name}_{oversam}_{tuning}_auc curve.png", bbox_inches = "tight")
    
    plt.figure(figsize=(10,10))
    for n in model_names:
        plt.plot(
            results[n]["recall"],
            results[n]["precision"],
            # where="post",
            label=f"{n} (PR-AUC = {results[n]['pr_auc']:.2f})"
        )

    plt.xlim([-0.05, 1.05])
    plt.ylim([-0.05, 1.05])
    plt.xlabel("Recall")
    plt.ylabel("Precision")
    plt.title(f"Precision-Recall Curves of OSA Prediction Models\n using {feature_name}")
    plt.tight_layout()
    plt.legend(loc="lower left")
    plt.savefig(f"renewal 1/{feature_name}_{oversam}_{tuning}_p-r curve.png", bbox_inches="tight")
    plt.close()

    plt.figure(figsize=(10,10))
    for n in model_names :
        plt.plot(results[n]["prob_pred"], results[n]["prob_true"], label=f"{n}")
    plt.plot([0,1], [0,1], "k--", label = "Perfectly Calibrated")
    plt.xlim([-0.05, 1.05])
    plt.ylim([-0.05, 1.05])
    plt.xlabel("Mean Predicted Probability")
    plt.ylabel("Fraction of Positives")
    plt.title(f"Calibration Curves of OSA Prediction Models \n using {feature_name}")
    plt.tight_layout()
    plt.legend()
    plt.savefig(f"renewal 1/{feature_name}_{oversam}_{tuning}_calibration curve.png", bbox_inches = "tight")
        
        
    return score

# data load

In [ ]:
trains = pd.read_csv("train_data.csv", index_col=0)
trains.rename(columns = {"first_recovery_mean" : "HR_first_recovery_phase", "second_recovery_mean" : "HR_second_recovery_phase","tense_steps" : "moderate-intensity exercise"}, inplace = True)
tests = pd.read_csv("test_data.csv", index_col=0)
tests.rename(columns = {"first_recovery_mean" : "HR_first_recovery_phase", "second_recovery_mean" : "HR_second_recovery_phase","tense_steps" : "moderate-intensity exercise"}, inplace = True)


In [ ]:
all_columns = trains.columns.tolist()
all_columns.remove("BERLIN01_index")
num_columns = trains._get_numeric_data().columns.to_list()
cat_columns = list(set(all_columns) - set(num_columns))


In [ ]:
scaler = StandardScaler()
scaled_num_trains = scaler.fit_transform(trains.copy()[num_columns])
num_trains_df = pd.DataFrame(scaled_num_trains, columns = num_columns, index = trains.index)
scaled_trains = pd.concat([num_trains_df, trains[cat_columns]], axis = 1)
scaled_num_tests = scaler.transform(tests.copy()[num_columns])
num_test_df = pd.DataFrame(scaled_num_tests, columns = num_columns, index = tests.index)
scaled_tests = pd.concat([num_test_df, tests[cat_columns]], axis = 1)
scaled_trains

# train and test

In [ ]:
custom_palette = {
    "Wearable Device": "#4E79A7",      
    "Smartphone App": "#F28E2B",           
    "Self-Questionnaire": "#76B7B2",     
    "Demographic Information": "#E15759" 
}
smt = SMOTE(random_state=42)
re_train_x, re_train_y = smt.fit_resample(scaled_trains[[x for x in trains if "BERLIN" not in x]],scaled_trains["BERLIN_index"])
feature_score = mutual_info_classif(re_train_x, re_train_y, random_state = 0)
selected_columns_index = np.where(feature_score>= np.percentile(feature_score,75))
selected_columns = f_x.columns[selected_columns_index]
len(selected_columns)

calls = ["Selected data", "All data", "Least data"]
cols = [selected_columns, [x for x in trains if "BERLIN" not in x], ["BMI", "Sex", "Age"]]
n_iters = 50
SK_cv = StratifiedKFold(n_splits=5, shuffle=True)
smt = SMOTE(random_state=42)
re_train_x, re_train_y = smt.fit_resample(scaled_trains[[x for x in trains if "BERLIN" not in x]],scaled_trains["BERLIN_index"])
for feature_name, use_columns in zip(calls, cols) :
    train_x, train_y = re_train_x[use_columns], re_train_y
    test_x, test_y = scaled_tests[use_columns], scaled_tests["BERLIN_index"]
    oversam = "SMOTE"
    tuning = "tuning"
    tuned_xgb = tuning_xgb()
    tuned_lgbm = tuning_lgbm()
    tuned_rf = tuning_rf()
    tuned_xgbrf = tuning_xgbrf()
    models = [tuned_xgb, tuned_lgbm, tuned_rf,tuned_xgbrf]        
    model_names = ["Extreme Gradient Boosting", "Light Gradient Boosting", "Random Forest", "XGBRF"]                
    result_score = model_run(train_x, train_y, test_x, test_y, models, model_names)
    pd.DataFrame(result_score).T.to_excel(f"renewal 1/{feature_name}_{oversam}_{tuning}_performance_renewal.xlsx")
    for m, n in zip(models, model_names) :
        oversam = "SMOTE"
        columns = feature_name
        title_name = feature_name
        plot_FI(train_x, train_y, test_x, m, n)